# Comparación de modelos con validación cruzada (7.3)

Dueño: Juanes. Solo análisis y figuras: la lógica reutilizable va en `src/`.
Antes de cada commit: Kernel → Restart & Run All.

In [ ]:
from src.config import RUTA_FIGURAS, RUTA_TABLAS, SEMILLA
from src.figuras import aplicar_estilo, guardar_figura

aplicar_estilo()


## Día 1 · Conocer los datos: la serie nacional de muebles

Solo entrenamiento (febrero de 2019 a julio de 2025). La serie nacional es la suma ponderada de los 7 departamentos con `PESOS_DEPTO`: es la misma cuenta que hará `agregar_nacional()` de Aleja y la serie sobre la que se mide el MAPE del artículo.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.config import LINEA_OBJETIVO, LINEA_OBJETIVO_CODIGO, OBJETIVO, PESOS_DEPTO
from src.datos.cargar import cargar_entrenamiento
from src.evaluacion.particion import CVTemporalPorMes

df = cargar_entrenamiento("central")
print(df.shape, df["fecha"].min().date(), "a", df["fecha"].max().date())


In [ ]:
muebles = df[df["linea_codigo"] == LINEA_OBJETIVO_CODIGO].copy()
muebles["aporte"] = muebles[OBJETIVO] * muebles["departamento"].map(PESOS_DEPTO)
serie_nacional = muebles.groupby("fecha")["aporte"].sum()
serie_nacional.describe().round(2)


Franjas grises: las 5 ventanas de validación de `CVTemporalPorMes` (F1 a F5). Franja roja: los meses de `covid`.


In [ ]:
ventanas = CVTemporalPorMes().ventanas(df)
meses_covid = df.loc[df["covid"] == 1, "fecha"].unique()

fig, ax = plt.subplots(figsize=(9, 3.6))
ax.plot(serie_nacional.index, serie_nacional.values, color="#1f4e79", lw=1.6,
        label="Índice nacional de muebles")
ax.axvspan(meses_covid.min(), meses_covid.max() + pd.DateOffset(months=1),
           color="#c0392b", alpha=0.12, label="covid (abr–ago 2020)")
for v in ventanas.itertuples():
    ax.axvspan(v.valida_desde, v.valida_hasta + pd.DateOffset(months=1),
               color="#7f8c8d", alpha=0.08 + 0.04 * (v.fold % 2))
    ax.text(v.valida_desde + pd.DateOffset(days=75), serie_nacional.max() * 1.02,
            f"F{v.fold}", ha="center", fontsize=8, color="#555555")
ax.set_title(f"{LINEA_OBJETIVO}: serie nacional (escenario central, solo entrenamiento)")
ax.set_ylabel("Índice de ventas reales (2019 = 100)")
ax.set_xlabel("Mes")
ax.legend(loc="lower right", fontsize=8)
guardar_figura(fig, "exploracion_serie_nacional_muebles")


**Lectura.** La caída de abril de 2020 (≈ 41) y el rebote posterior son el mayor atípico de la serie y quedan siempre en entrenamiento, nunca en validación. La estacionalidad es clara: noviembre y diciembre son los meses más altos y abril el más bajo (en promedio), con saltos puntuales como junio de 2022. Desde 2023 la serie es más estable y en 2025 crece: los modelos deben capturar esa tendencia y el pico de fin de año.


## Lunes 26 · Comparación de modelos con validación cruzada (figura 3)

Lee `results/tablas/cv_resultados.csv`, que deja `python scripts/entrenar.py --escenario todos`. Cada punto es el MAPE nacional de validación de un fold (5 por modelo) y el rombo es su media. Los modelos van de mejor (arriba) a peor (abajo) en cada escenario. La línea roja es el MAPE medio del ingenuo estacional: lo que queda a su izquierda lo supera en promedio. Mirar los 5 puntos y no solo el rombo muestra la variabilidad entre periodos que pide la 7.4.


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

from src.config import ESCENARIOS

cv = pd.read_csv(RUTA_TABLAS / "cv_resultados.csv")

# Nombres legibles para el informe
NOMBRES = {
    "media": "Media (base)",
    "ingenuo_estacional": "Ingenuo estacional (base)",
    "mlr_stepwise": "MLR stepwise",
    "mlr_completa": "MLR completa",
    "ridge": "Ridge",
    "lasso": "Lasso",
    "knn": "KNN",
}
BASES = {"media", "ingenuo_estacional"}

fig, ejes = plt.subplots(1, len(ESCENARIOS), figsize=(10, 3.6), sharex=True)
for ax, escenario in zip(ejes, ["bajo", "central", "alto"]):
    datos = cv[cv["escenario"] == escenario]
    # De mejor (arriba) a peor (abajo) por MAPE medio de validación
    orden = datos.groupby("modelo")["mape"].mean().sort_values(ascending=False)
    for y, (modelo, media) in enumerate(orden.items()):
        folds = datos.loc[datos["modelo"] == modelo, "mape"]
        color = "#7f8c8d" if modelo in BASES else "#1f4e79"
        ax.scatter(folds, [y] * len(folds), s=14, color=color, alpha=0.45, zorder=2)
        ax.scatter([media], [y], s=60, marker="D", color=color, edgecolor="white", zorder=3)
    referencia = orden["ingenuo_estacional"]
    ax.axvline(referencia, color="#c0392b", ls="--", lw=1, zorder=1)
    ax.set_yticks(range(len(orden)), [NOMBRES[m] for m in orden.index], fontsize=8)
    ax.set_title(f"Escenario {escenario}", fontsize=10)
    ax.set_xlabel("MAPE nacional de validación (%)")
    ax.grid(axis="y", alpha=0)
fig.suptitle("MAPE por fold (puntos) y media de los 5 folds (rombo), de mejor a peor",
             fontsize=10)
fig.tight_layout(rect=(0, 0.05, 1, 1))
fig.text(0.5, 0.01, "Línea roja punteada: MAPE medio del ingenuo estacional (la vara exigente). "
         "Gris: líneas base; azul: modelos.", ha="center", fontsize=8, color="#555555")
guardar_figura(fig, "fig03_comparacion_cv")


**Lectura (escenario central).** Ridge (9,1 %), Lasso (9,4 %) y KNN (9,0 %) superan a las dos líneas base; la MLR, con o sin stepwise (14,8 % y 15,0 %), queda peor que ambas. La media (10,3 %) supera al ingenuo estacional (12,3 %), contra lo que se esperaba. El fold 5 (febrero a julio de 2025) es el más difícil para todos (16 % a 21 %): la serie crece en 2025 y ningún modelo lo anticipa. El orden de los mejores cambia poco entre escenarios, salvo KNN, que en el escenario alto cae al nivel de la media (10,3 %).
